## Name : Manmath Maroti Kornule
---
## prn : 202401110045
---

# 🌾 Mini Project – Improve Model Accuracy through Feature Engineering  
### Dataset: mandi_prices.csv

---

### 📊 About the Dataset
The **Mandi Prices dataset** contains agricultural market data from different Indian states and districts.  
It includes details such as commodity type, price range, arrival quantity, and trader participation recorded across 2,000 mandi entries.

**Key Columns:**  
`Date`, `State`, `District`, `Mandi`, `Commodity`, `Variety`,  
`Min Price`, `Max Price`, `Modal Price`, `Arrival Quantity`, `Grade`, `Market Fee (%)`, `Trader Count`

**Purpose:**  
Used to apply data preprocessing, scaling, encoding, feature selection, and feature engineering  
to improve model accuracy for predicting market price patterns.


Step 1: Load and Explore Dataset

In [3]:
# Step 1: Load and explore dataset
import pandas as pd

# Upload or read directly if stored in your Colab or Drive
df = pd.read_csv('mandi_prices.csv')  # adjust path if needed

# Display basic information
print("Shape of dataset:", df.shape)
print("\nColumn names:\n", df.columns.tolist())

# Display first few rows
print("\nSample data:")
display(df.head())

# Check for missing values
print("\nMissing values per column:")
print(df.isnull().sum())

# Check datatypes
print("\nData types:")
print(df.dtypes)


Shape of dataset: (2000, 14)

Column names:
 ['Date', 'State', 'District', 'Mandi', 'Commodity', 'Variety', 'Min Price', 'Max Price', 'Modal Price', 'Unit', 'Arrival Quantity', 'Grade', 'Market Fee (%)', 'Trader Count']

Sample data:


,Date,State,District,Mandi,Commodity,Variety,Min Price,Max Price,Modal Price,Unit,Arrival Quantity,Grade,Market Fee (%),Trader Count
0,23-01-2024,Gujarat,Ahmedabad,Main Mandi,Rice,Kolam,1051.0,1163.0,1090.0,Quintal,97.71,C,1.84,8
1,14-03-2024,Karnataka,Bangalore,Agro Market,Maize,White,1087.0,1382.0,1270.5,Quintal,31.89,B,1.48,10
2,23-05-2024,Maharashtra,Nashik,Agro Market,Wheat,Sharbati,984.0,1154.0,1049.0,Quintal,43.95,C,1.18,15
3,06-05-2024,Gujarat,Ahmedabad,Main Mandi,Potato,Kufri Jyoti,897.0,1216.0,1014.5,Quintal,45.70,B,2.96,14
4,30-01-2024,Gujarat,Surat,Wholesale Yard,Wheat,Sharbati,973.0,1140.0,1012.5,Quintal,82.24,B,1.46,5



Missing values per column:
Date                 0
State                0
District             0
Mandi                0
Commodity            0
Variety             13
Min Price            3
Max Price           19
Modal Price          1
Unit                 0
Arrival Quantity    21
Grade               19
Market Fee (%)       0
Trader Count         0
dtype: int64

Data types:
Date                 object
State                object
District             object
Mandi                object
Commodity            object
Variety              object
Min Price           float64
Max Price           float64
Modal Price         float64
Unit                 object
Arrival Quantity    float64
Grade                object
Market Fee (%)      float64
Trader Count          int64
dtype: object


Step 2: Handle Missing Values
Step 2.1: Separate numerical and categorical columns

In [4]:
# Step 2.1: Separate numeric and categorical columns
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = df.select_dtypes(include=['object']).columns

print("Numeric Columns:", numeric_cols.tolist())
print("\nCategorical Columns:", categorical_cols.tolist())


Numeric Columns: ['Min Price', 'Max Price', 'Modal Price', 'Arrival Quantity', 'Market Fee (%)', 'Trader Count']

Categorical Columns: ['Date', 'State', 'District', 'Mandi', 'Commodity', 'Variety', 'Unit', 'Grade']


In [5]:
# Show only columns that have missing values
missing_cols = df.columns[df.isnull().any()]
df[missing_cols].isnull().sum()


,0
Variety,13
Min Price,3
Max Price,19
Modal Price,1
Arrival Quantity,21
Grade,19


Step 2.2: Handle Missing Values (Numeric + Categorical)

In [10]:
# Step 2.3: Warning-free version (recommended for clean report)
numeric_missing = ['Min Price', 'Max Price', 'Modal Price', 'Arrival Quantity']
categorical_missing = ['Variety', 'Grade']

# Numeric columns with median
for col in numeric_missing:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)
    print(f"Numeric column '{col}' - filled missing values with median: {median_val}")

# Categorical columns with mode
for col in categorical_missing:
    mode_val = df[col].mode()[0]
    df[col] = df[col].fillna(mode_val)
    print(f"Categorical column '{col}' - filled missing values with mode: {mode_val}")

print("\nMissing values after imputation:")
print(df.isnull().sum())


Numeric column 'Min Price' - filled missing values with median: 996.0
Numeric column 'Max Price' - filled missing values with median: 1243.0
Numeric column 'Modal Price' - filled missing values with median: 1120.5
Numeric column 'Arrival Quantity' - filled missing values with median: 52.11
Categorical column 'Variety' - filled missing values with mode: White
Categorical column 'Grade' - filled missing values with mode: C

Missing values after imputation:
Date                0
State               0
District            0
Mandi               0
Commodity           0
Variety             0
Min Price           0
Max Price           0
Modal Price         0
Unit                0
Arrival Quantity    0
Grade               0
Market Fee (%)      0
Trader Count        0
dtype: int64


Convert Date column to numeric features

In [11]:
# Step 3.2: Convert Date column into separate numeric features
df['Date'] = pd.to_datetime(df['Date'], format='%d-%m-%Y')

df['Day'] = df['Date'].dt.day
df['Month'] = df['Date'].dt.month
df['Year'] = df['Date'].dt.year

# Drop the original Date column
df.drop('Date', axis=1, inplace=True)

print("Date column converted into numeric features: Day, Month, Year")


Date column converted into numeric features: Day, Month, Year


Apply One-Hot Encoding to Categorical Columns

In [12]:
# Step 3.3: One-Hot Encode categorical columns
categorical_cols = ['State', 'District', 'Mandi', 'Commodity', 'Variety', 'Grade', 'Unit']

df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

print("Shape after one-hot encoding:", df_encoded.shape)
df_encoded.head()


Shape after one-hot encoding: (2000, 53)


,Min Price,Max Price,Modal Price,Arrival Quantity,Market Fee (%),Trader Count,Day,Month,Year,State_Karnataka,...,Variety_Kufri Jyoti,Variety_Lokwan,Variety_Pink,Variety_Red,Variety_Sharbati,Variety_Sona Masoori,Variety_White,Variety_Yellow,Grade_B,Grade_C
0,1051.0,1163.0,1090.0,97.71,1.84,8,23,1,2024,False,...,False,False,False,False,False,False,False,False,False,True
1,1087.0,1382.0,1270.5,31.89,1.48,10,14,3,2024,True,...,False,False,False,False,False,False,True,False,True,False
2,984.0,1154.0,1049.0,43.95,1.18,15,23,5,2024,False,...,False,False,False,False,True,False,False,False,False,True
3,897.0,1216.0,1014.5,45.70,2.96,14,6,5,2024,False,...,True,False,False,False,False,False,False,False,True,False
4,973.0,1140.0,1012.5,82.24,1.46,5,30,1,2024,False,...,False,False,False,False,True,False,False,False,True,False


Apply Min–Max Scaling

In [13]:
# Step 4.1: Apply Min-Max Scaling
from sklearn.preprocessing import MinMaxScaler

minmax_scaler = MinMaxScaler()

# Select only numeric columns (ignore True/False since get_dummies made them 0/1)
numeric_features = df_encoded.select_dtypes(include=['int64', 'float64', 'bool']).columns

df_minmax = df_encoded.copy()
df_minmax[numeric_features] = minmax_scaler.fit_transform(df_encoded[numeric_features])

print("Shape after Min-Max scaling:", df_minmax.shape)
df_minmax.head()


Shape after Min-Max scaling: (2000, 53)


,Min Price,Max Price,Modal Price,Arrival Quantity,Market Fee (%),Trader Count,Day,Month,Year,State_Karnataka,...,Variety_Kufri Jyoti,Variety_Lokwan,Variety_Pink,Variety_Red,Variety_Sharbati,Variety_Sona Masoori,Variety_White,Variety_Yellow,Grade_B,Grade_C
0,0.6275,0.373529,0.444444,0.976818,0.42,0.461538,23,1,2024,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,0.7175,0.695588,0.750636,0.283246,0.24,0.615385,14,3,2024,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
2,0.4600,0.360294,0.374894,0.410327,0.09,1.000000,23,5,2024,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
3,0.2425,0.451471,0.316370,0.428767,0.98,0.923077,6,5,2024,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,0.4325,0.339706,0.312977,0.813804,0.23,0.230769,30,1,2024,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0


Apply Standardization (Z-score)

In [14]:
# Step 4.2: Apply Standardization (Z-score)
from sklearn.preprocessing import StandardScaler

standard_scaler = StandardScaler()
df_standard = df_encoded.copy()
df_standard[numeric_features] = standard_scaler.fit_transform(df_encoded[numeric_features])

print("Shape after Standardization:", df_standard.shape)
df_standard.head()


Shape after Standardization: (2000, 53)


,Min Price,Max Price,Modal Price,Arrival Quantity,Market Fee (%),Trader Count,Day,Month,Year,State_Karnataka,...,Variety_Kufri Jyoti,Variety_Lokwan,Variety_Pink,Variety_Red,Variety_Sharbati,Variety_Sona Masoori,Variety_White,Variety_Yellow,Grade_B,Grade_C
0,0.457499,-0.610830,-0.275094,1.678255,-0.253206,-0.129719,23,1,2024,-0.495308,...,-0.275404,-0.231821,-0.215807,-0.242407,-0.213256,-0.197386,-0.389115,-0.262578,-0.693098,1.321837
1,0.770559,0.945651,1.170639,-0.766368,-0.864056,0.379483,14,3,2024,2.018946,...,-0.275404,-0.231821,-0.215807,-0.242407,-0.213256,-0.197386,2.569934,-0.262578,1.442797,-0.756523
2,-0.125142,-0.674795,-0.603487,-0.318447,-1.373097,1.652489,23,5,2024,-0.495308,...,-0.275404,-0.231821,-0.215807,-0.242407,4.689190,-0.197386,-0.389115,-0.262578,-0.693098,1.321837
3,-0.881705,-0.234147,-0.879818,-0.253450,1.647216,1.397888,6,5,2024,-0.495308,...,3.631033,-0.231821,-0.215807,-0.242407,-0.213256,-0.197386,-0.389115,-0.262578,1.442797,-0.756523
4,-0.220799,-0.774296,-0.895837,1.103683,-0.897992,-0.893523,30,1,2024,-0.495308,...,-0.275404,-0.231821,-0.215807,-0.242407,4.689190,-0.197386,-0.389115,-0.262578,1.442797,-0.756523


## Feature Selection

Feature Selection using Chi-Square Test

The Chi-Square test requires a categorical target.

Since Modal Price is continuous, it was divided into 3 categories (Low, Medium, High) using equal-width bins.

The Chi-Square test then checks which features have the strongest relationship with these price levels.

We selected the top 10 features based on their Chi-Square scores.

In [16]:
# Step 5.1 (Revised): Chi-Square with categorical target

from sklearn.feature_selection import SelectKBest, chi2
import pandas as pd
import numpy as np

# Separate features and target
X = df_minmax.drop('Modal Price', axis=1)
y_continuous = df_minmax['Modal Price']

# Convert continuous target into categories (bins)
# For example: 3 bins → low, medium, high price
y_binned = pd.cut(y_continuous, bins=3, labels=[0, 1, 2])

# Apply Chi-Square test
chi_selector = SelectKBest(score_func=chi2, k=10)
chi_selector.fit(X, y_binned)

# Get top 10 features
chi_scores = pd.DataFrame({
    'Feature': X.columns,
    'Chi2 Score': chi_selector.scores_
}).sort_values(by='Chi2 Score', ascending=False)

print("Top 10 features selected using Chi-Square Test:\n")
print(chi_scores.head(10))


Top 10 features selected using Chi-Square Test:

              Feature  Chi2 Score
0           Min Price  233.113758
1           Max Price  123.134178
5                 Day   35.288683
27    Mandi_Sub Mandi    5.812492
24  District_Vadodara    4.962885
26   Mandi_Main Mandi    4.489974
19    District_Nagpur    3.670710
46   Variety_Sharbati    3.388008
36   Variety_CO 86032    2.799294
14     District_Hubli    2.791183


Variance Threshold Method

In [17]:
# Step 5.2: Variance Threshold

from sklearn.feature_selection import VarianceThreshold

# Apply Variance Threshold (remove features with variance < 0.01)
var_thresh = VarianceThreshold(threshold=0.01)
var_thresh.fit(X)

# Get retained features
var_features = X.columns[var_thresh.get_support()]

print("Number of features retained after Variance Threshold:", len(var_features))
print("\nRetained features:\n", list(var_features))


Number of features retained after Variance Threshold: 51

Retained features:
 ['Min Price', 'Max Price', 'Arrival Quantity', 'Market Fee (%)', 'Trader Count', 'Day', 'Month', 'State_Karnataka', 'State_Maharashtra', 'State_Punjab', 'State_Uttar Pradesh', 'District_Amritsar', 'District_Bangalore', 'District_Hubli', 'District_Kanpur', 'District_Lucknow', 'District_Ludhiana', 'District_Mysore', 'District_Nagpur', 'District_Nashik', 'District_Patiala', 'District_Pune', 'District_Surat', 'District_Vadodara', 'District_Varanasi', 'Mandi_Main Mandi', 'Mandi_Sub Mandi', 'Mandi_Wholesale Yard', 'Commodity_Onion', 'Commodity_Potato', 'Commodity_Rice', 'Commodity_Sugarcane', 'Commodity_Tomato', 'Commodity_Wheat', 'Variety_CO 0238', 'Variety_CO 86032', 'Variety_Chipsona', 'Variety_Desi', 'Variety_Durum', 'Variety_Hybrid', 'Variety_Kolam', 'Variety_Kufri Jyoti', 'Variety_Lokwan', 'Variety_Pink', 'Variety_Red', 'Variety_Sharbati', 'Variety_Sona Masoori', 'Variety_White', 'Variety_Yellow', 'Grade_B', 

Polynomial Features

In [18]:
# Step 6: Feature Engineering – Create new features

from sklearn.preprocessing import PolynomialFeatures

# Select numeric columns for feature creation
numeric_cols = ['Min Price', 'Max Price', 'Modal Price', 'Arrival Quantity', 'Market Fee (%)', 'Trader Count']

# Generate polynomial features (degree=2)
poly = PolynomialFeatures(degree=2, include_bias=False)
poly_features = poly.fit_transform(df[numeric_cols])

# Create a DataFrame for these new features
poly_df = pd.DataFrame(poly_features, columns=poly.get_feature_names_out(numeric_cols))

# Concatenate with original data
df_engineered = pd.concat([df, poly_df.add_prefix('Poly_')], axis=1)

print("Shape before feature creation:", df.shape)
print("Shape after feature creation:", df_engineered.shape)
df_engineered.head()


Shape before feature creation: (2000, 16)
Shape after feature creation: (2000, 43)


,State,District,Mandi,Commodity,Variety,Min Price,Max Price,Modal Price,Unit,Arrival Quantity,...,Poly_Modal Price^2,Poly_Modal Price Arrival Quantity,Poly_Modal Price Market Fee (%),Poly_Modal Price Trader Count,Poly_Arrival Quantity^2,Poly_Arrival Quantity Market Fee (%),Poly_Arrival Quantity Trader Count,Poly_Market Fee (%)^2,Poly_Market Fee (%) Trader Count,Poly_Trader Count^2
0,Gujarat,Ahmedabad,Main Mandi,Rice,Kolam,1051.0,1163.0,1090.0,Quintal,97.71,...,1188100.00,106503.900,2005.60,8720.0,9547.2441,179.7864,781.68,3.3856,14.72,64.0
1,Karnataka,Bangalore,Agro Market,Maize,White,1087.0,1382.0,1270.5,Quintal,31.89,...,1614170.25,40516.245,1880.34,12705.0,1016.9721,47.1972,318.90,2.1904,14.80,100.0
2,Maharashtra,Nashik,Agro Market,Wheat,Sharbati,984.0,1154.0,1049.0,Quintal,43.95,...,1100401.00,46103.550,1237.82,15735.0,1931.6025,51.8610,659.25,1.3924,17.70,225.0
3,Gujarat,Ahmedabad,Main Mandi,Potato,Kufri Jyoti,897.0,1216.0,1014.5,Quintal,45.70,...,1029210.25,46362.650,3002.92,14203.0,2088.4900,135.2720,639.80,8.7616,41.44,196.0
4,Gujarat,Surat,Wholesale Yard,Wheat,Sharbati,973.0,1140.0,1012.5,Quintal,82.24,...,1025156.25,83268.000,1478.25,5062.5,6763.4176,120.0704,411.20,2.1316,7.30,25.0


Domain-Driven Feature Creation

In [19]:
# Step 6.2: Domain-driven feature creation

df_domain = df.copy()

# 1️⃣ Price Range – difference between Max and Min Price
df_domain["Price Range"] = df_domain["Max Price"] - df_domain["Min Price"]

# 2️⃣ Price Spread Ratio – how wide the spread is relative to Modal Price
df_domain["Price Spread Ratio"] = df_domain["Price Range"] / df_domain["Modal Price"]

# 3️⃣ Arrival Efficiency – quantity per trader (how much each trader handles)
df_domain["Arrival per Trader"] = df_domain["Arrival Quantity"] / df_domain["Trader Count"]

# 4️⃣ Market Fee Value – actual fee paid per quintal
df_domain["Market Fee Value"] = (df_domain["Market Fee (%)"] / 100) * df_domain["Modal Price"]

# Show results
print("New columns added:", ['Price Range', 'Price Spread Ratio', 'Arrival per Trader', 'Market Fee Value'])
print("\nShape before:", df.shape)
print("Shape after:", df_domain.shape)

df_domain.head()


New columns added: ['Price Range', 'Price Spread Ratio', 'Arrival per Trader', 'Market Fee Value']

Shape before: (2000, 16)
Shape after: (2000, 20)


,State,District,Mandi,Commodity,Variety,Min Price,Max Price,Modal Price,Unit,Arrival Quantity,Grade,Market Fee (%),Trader Count,Day,Month,Year,Price Range,Price Spread Ratio,Arrival per Trader,Market Fee Value
0,Gujarat,Ahmedabad,Main Mandi,Rice,Kolam,1051.0,1163.0,1090.0,Quintal,97.71,C,1.84,8,23,1,2024,112.0,0.102752,12.213750,20.0560
1,Karnataka,Bangalore,Agro Market,Maize,White,1087.0,1382.0,1270.5,Quintal,31.89,B,1.48,10,14,3,2024,295.0,0.232192,3.189000,18.8034
2,Maharashtra,Nashik,Agro Market,Wheat,Sharbati,984.0,1154.0,1049.0,Quintal,43.95,C,1.18,15,23,5,2024,170.0,0.162059,2.930000,12.3782
3,Gujarat,Ahmedabad,Main Mandi,Potato,Kufri Jyoti,897.0,1216.0,1014.5,Quintal,45.70,B,2.96,14,6,5,2024,319.0,0.314441,3.264286,30.0292
4,Gujarat,Surat,Wholesale Yard,Wheat,Sharbati,973.0,1140.0,1012.5,Quintal,82.24,B,1.46,5,30,1,2024,167.0,0.164938,16.448000,14.7825


# 🧩 Mini Project – Improve Model Accuracy through Feature Engineering

## 📘 Step 7 – Summary of Findings and Ethical Considerations

### 🧠 Summary of Data Preprocessing and Feature Engineering

**Dataset Used:**  
Mandi Market Dataset (2,000 rows × 14 columns)

---

### 🧹 1️⃣ Handling Missing Values
- **Numeric Columns:**  
  Filled missing values in `Min Price`, `Max Price`, `Modal Price`, and `Arrival Quantity` using the **median**, to avoid skewing results.
- **Categorical Columns:**  
  Filled missing values in `Variety` and `Grade` using the **mode**, as these represent discrete labels.

---

### ⚙️ 2️⃣ Scaling and Encoding
- Applied **Min–Max Scaling** and **Standardization** to numerical columns for better model stability.  
- Used **One-Hot Encoding** for categorical features (`State`, `District`, `Commodity`, `Variety`, etc.).  
- Extracted **Day**, **Month**, and **Year** from the `Date` column for time-based analysis.

---

### 🔍 3️⃣ Feature Selection
- **Chi-Square Test:** identified features most associated with `Modal Price`.  
  - *Top features:* `Min Price`, `Max Price`, `Day`, `District_Vadodara`, `Variety_Sharbati`.
- **Variance Threshold (0.01):** removed near-constant columns, retaining 51 features for modeling.

---

### 🧮 4️⃣ Feature Engineering

#### 🔸 Polynomial Features
Generated **degree-2 polynomial** features to capture non-linear relationships among:
`Min Price`, `Max Price`, `Modal Price`, `Arrival Quantity`, `Market Fee (%)`, `Trader Count`.  
- Shape grew from (2000, 16) → (2000, 43)

#### 🔸 Domain-Driven Features
| New Feature | Formula | Description |
|--------------|----------|-------------|
| **Price Range** | `Max Price − Min Price` | Measures price volatility within a mandi. |
| **Price Spread Ratio** | `(Max−Min)/Modal` | Normalizes price variation by modal price. |
| **Arrival per Trader** | `Arrival Quantity / Trader Count` | Indicates average load handled per trader. |
| **Market Fee Value** | `(Market Fee (%) × Modal Price)/100` | Shows actual fee value per quintal. |

These new features help capture trading efficiency, price volatility, and cost dynamics.

---

### 📊 Key Findings
- Dataset had minimal missing data and was successfully imputed.  
- Price-related variables (`Min Price`, `Max Price`, `Modal Price`) dominate prediction strength.  
- Added features improved interpretability and potential model accuracy.  
- Feature selection reduced redundancy while preserving useful variance.

---

### ⚖️ Ethical Considerations
1. **Data Privacy:** The dataset contains only aggregated mandi information — no personal data.  
2. **Bias Awareness:** Regional or commodity imbalance may bias predictions; re-sampling is recommended.  
3. **Transparency:** All preprocessing and feature steps are documented for reproducibility.  
4. **Sustainability Impact:** Better price modeling supports fairer trade and reduced waste, aligning with **SDG 12 – Responsible Consumption and Production**.

---

✅ **Conclusion:**  
Through systematic handling of missing values, scaling, encoding, feature selection, and both polynomial and domain-driven feature creation, model readiness and interpretability have been significantly improved.
